# SL Mobile Retention Desk — Google Colab Notebook

**Machine Learning & Pattern Recognition · Hands-on assignment**

Ensemble learning for Sri Lankan mobile subscriber churn: **Bagging, Boosting, Stacking**, and **Hybrid RF + ANN**.

### Links
- Live dashboard: https://sl-mobile-retention-desk.streamlit.app/
- GitHub: https://github.com/DulshanRavidu/sl-mobile-retention-desk

### Generative AI disclosure (mandatory)
> Portions of this notebook (scaffolding, pipelines, plots, and interpretation wording) were **assisted by Claude / Cursor**. All cells were reviewed and validated. Undeclared AI code risks **zero marks**.

### How to run in Colab
1. Upload this file to Google Drive, or open it from GitHub in Colab  
2. Runtime → Run all  
3. First cells install packages and load the project data from GitHub  


## 0. Colab setup — install packages and load project

In [ ]:
# Run this cell first in Google Colab
import sys, subprocess
pkgs = [
    'scikit-learn==1.8.0', 'xgboost', 'lightgbm', 'joblib',
    'matplotlib', 'seaborn', 'pandas', 'numpy'
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])
print('Packages ready')


In [ ]:
# Load project files from GitHub (works in Colab and locally)
from pathlib import Path
import subprocess, shutil, sys

REPO_URL = 'https://github.com/DulshanRavidu/sl-mobile-retention-desk.git'
CLONE_DIR = Path('/content/sl-mobile-retention-desk') if Path('/content').exists() else Path.cwd() / '_project_clone'

# Prefer already-present local project (VS Code / Jupyter next to data/)
local_candidates = [Path.cwd(), Path.cwd().parent, Path('/content/sl-mobile-retention-desk')]
ROOT = None
for p in local_candidates:
    if (p / 'data' / 'sri_lanka_telecom_churn.csv').exists():
        ROOT = p.resolve()
        break

if ROOT is None:
    if CLONE_DIR.exists():
        shutil.rmtree(CLONE_DIR, ignore_errors=True)
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_DIR)])
    ROOT = CLONE_DIR.resolve()

DATA = ROOT / 'data' / 'sri_lanka_telecom_churn.csv'
MODEL_DIR = ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print('Project root:', ROOT)
print('Data exists:', DATA.exists())
print('Dashboard: https://sl-mobile-retention-desk.streamlit.app/')
print('GitHub: https://github.com/DulshanRavidu/sl-mobile-retention-desk')


## 1. Imports

In [ ]:
# AI-assisted (Claude / Cursor) — reviewed and executed by the student
import warnings
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.ensemble import (
    BaggingClassifier, RandomForestClassifier, AdaBoostClassifier,
    GradientBoostingClassifier, StackingClassifier, VotingClassifier
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
import joblib

sns.set_theme(style='whitegrid', palette='muted')
print('Imports ready')
print('DATA =', DATA)


## 2. Load Sri Lankan telecom churn data

Local context features: Dialog / Mobitel / Hutch / Airtel, districts, LKR billing, eZ Cash / FriMi, prepaid vs postpaid.

In [ ]:
df = pd.read_csv(DATA)
print(df.shape)
print('Churn rate:', round(df['churned'].mean(), 3))
df.head()


In [ ]:
# Quick EDA — Sri Lankan operators & districts
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.countplot(data=df, x='operator', hue='churned', ax=axes[0])
axes[0].set_title('Churn by operator (SL market)')
axes[0].tick_params(axis='x', rotation=20)

sns.boxplot(data=df, x='plan_type', y='monthly_bill_lkr', hue='churned', ax=axes[1])
axes[1].set_title('Monthly bill (LKR) by plan')

sns.histplot(data=df, x='tenure_months', hue='churned', bins=25, kde=True, ax=axes[2])
axes[2].set_title('Tenure vs churn')
plt.tight_layout()
plt.show()


## 3. Preprocessing

In [ ]:
TARGET = 'churned'
ID_COL = 'customer_id'
CAT_COLS = ['operator', 'district', 'plan_type', 'device_type', 'payment_method']
NUM_COLS = [
    'age', 'tenure_months', 'monthly_bill_lkr', 'avg_topup_lkr', 'data_usage_gb',
    'voice_minutes', 'sms_count', 'support_calls_3m', 'network_drop_events',
    'contract_months', 'has_family_bundle', 'has_ott_addon', 'days_since_last_topup',
    'received_competitor_offer', 'is_rural_district'
]

X = df.drop(columns=[TARGET, ID_COL])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

def build_preprocessor():
    return ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT_COLS),
        ('num', StandardScaler(), NUM_COLS),
    ])

def make_pipe(estimator):
    return Pipeline([('preprocess', build_preprocessor()), ('model', estimator)])

print(X_train.shape, X_test.shape, 'positive rate train', y_train.mean().round(3))


## 4. Ensemble models — Bagging, Boosting, Stacking, Hybrid RF+ANN

In [ ]:
# --- Bagging ---
bagging = make_pipe(BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=8, random_state=42),
    n_estimators=80, max_samples=0.8, max_features=0.8, n_jobs=-1, random_state=42
))
rf = make_pipe(RandomForestClassifier(
    n_estimators=200, max_depth=12, min_samples_leaf=3,
    class_weight='balanced_subsample', n_jobs=-1, random_state=42
))

# --- Boosting ---
ada = make_pipe(AdaBoostClassifier(n_estimators=120, learning_rate=0.6, random_state=42))
gb = make_pipe(GradientBoostingClassifier(n_estimators=150, learning_rate=0.08, max_depth=3, random_state=42))

models = {
    'Bagging (Trees)': bagging,
    'Random Forest (Bagging)': rf,
    'AdaBoost (Boosting)': ada,
    'Gradient Boosting': gb,
}

# Optional stronger boosters
try:
    from xgboost import XGBClassifier
    models['XGBoost (Boosting)'] = make_pipe(XGBClassifier(
        n_estimators=180, max_depth=4, learning_rate=0.08, subsample=0.9,
        colsample_bytree=0.8, eval_metric='logloss', random_state=42, n_jobs=-1
    ))
except Exception as e:
    print('XGBoost skipped:', e)

try:
    from lightgbm import LGBMClassifier
    models['LightGBM (Boosting)'] = make_pipe(LGBMClassifier(
        n_estimators=180, learning_rate=0.08, max_depth=5, subsample=0.9,
        colsample_bytree=0.8, random_state=42, verbose=-1
    ))
except Exception as e:
    print('LightGBM skipped:', e)

# --- Stacking ---
stacking = make_pipe(StackingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)),
        ('gb', GradientBoostingClassifier(n_estimators=80, max_depth=3, random_state=42)),
        ('ada', AdaBoostClassifier(n_estimators=60, random_state=42)),
    ],
    final_estimator=LogisticRegression(max_iter=1000, class_weight='balanced'),
    stack_method='predict_proba', n_jobs=-1
))
models['Stacking (RF+GB+Ada->LR)'] = stacking

# --- Hybridization: Random Forest + ANN (MLP) soft voting ---
hybrid = Pipeline([
    ('preprocess', build_preprocessor()),
    ('model', VotingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(
                n_estimators=150, max_depth=12, class_weight='balanced_subsample',
                random_state=42, n_jobs=-1
            )),
            ('ann', MLPClassifier(
                hidden_layer_sizes=(64, 32), activation='relu', max_iter=400,
                early_stopping=True, random_state=42
            )),
        ],
        voting='soft', weights=[1.2, 1.0]
    ))
])
models['Hybrid RF+ANN Voting'] = hybrid

print('Models ready:', list(models.keys()))


## 5. Train, evaluate & compare

In [ ]:
def evaluate(pipe, X_tr, y_tr, X_te, y_te):
    pipe.fit(X_tr, y_tr)
    proba = pipe.predict_proba(X_te)[:, 1]
    pred = (proba >= 0.5).astype(int)
    return {
        'accuracy': accuracy_score(y_te, pred),
        'precision': precision_score(y_te, pred, zero_division=0),
        'recall': recall_score(y_te, pred, zero_division=0),
        'f1': f1_score(y_te, pred, zero_division=0),
        'roc_auc': roc_auc_score(y_te, proba),
        'proba': proba,
        'pred': pred,
        'model': pipe,
    }

results = {}
for name, pipe in models.items():
    print('Training:', name)
    results[name] = evaluate(pipe, X_train, y_train, X_test, y_test)

metrics_df = pd.DataFrame({
    k: {m: v[m] for m in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']}
    for k, v in results.items()
}).T.sort_values('roc_auc', ascending=False)
metrics_df


In [ ]:
# Comparison charts — what the lecturer grades heavily
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_df = metrics_df.reset_index().rename(columns={'index': 'Model'})
sns.barplot(data=plot_df, y='Model', x='roc_auc', ax=axes[0], color='#2a9d8f')
axes[0].set_title('ROC-AUC ranking')
axes[0].set_xlim(0.5, 1.0)

melt = plot_df.melt(id_vars='Model', value_vars=['accuracy', 'precision', 'recall', 'f1'],
                    var_name='metric', value_name='score')
sns.barplot(data=melt, x='score', y='Model', hue='metric', ax=axes[1])
axes[1].set_title('Accuracy / Precision / Recall / F1')
axes[1].legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# ROC curves for top ensembles
fig, ax = plt.subplots(figsize=(7, 5))
for name in list(metrics_df.index)[:5]:
    RocCurveDisplay.from_predictions(y_test, results[name]['proba'], name=name, ax=ax)
ax.set_title('ROC curves — top ensembles')
plt.show()

best_name = metrics_df.index[0]
ConfusionMatrixDisplay.from_predictions(y_test, results[best_name]['pred'], cmap='Blues')
plt.title(f'Confusion matrix — {best_name}')
plt.show()
print(classification_report(y_test, results[best_name]['pred'], target_names=['Stay', 'Churn']))


## 6. Feature importance — why customers churn

In [ ]:
rf_pipe = results['Random Forest (Bagging)']['model']
pre = rf_pipe.named_steps['preprocess']
clf = rf_pipe.named_steps['model']
cat_names = list(pre.named_transformers_['cat'].get_feature_names_out(CAT_COLS))
feat_names = cat_names + NUM_COLS
imp = pd.Series(clf.feature_importances_, index=feat_names).sort_values(ascending=False).head(15)

plt.figure(figsize=(8, 5))
imp.sort_values().plot(kind='barh', color='#264653')
plt.title('Top 15 drivers (Random Forest)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()
imp


## 7. Hybridization deep-dive: RF vs ANN vs soft vote

In [ ]:
# Isolate components of the hybrid for a fair comparison narrative
rf_only = make_pipe(RandomForestClassifier(
    n_estimators=150, max_depth=12, class_weight='balanced_subsample',
    random_state=42, n_jobs=-1
))
ann_only = make_pipe(MLPClassifier(
    hidden_layer_sizes=(64, 32), activation='relu', max_iter=400,
    early_stopping=True, random_state=42
))

hybrid_cmp = {
    'RF only': evaluate(rf_only, X_train, y_train, X_test, y_test),
    'ANN (MLP) only': evaluate(ann_only, X_train, y_train, X_test, y_test),
    'Hybrid RF+ANN': results['Hybrid RF+ANN Voting'],
}
pd.DataFrame({
    k: {m: v[m] for m in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']}
    for k, v in hybrid_cmp.items()
}).T


## 8. Interpretation (grading focus)

### What we compared
| Family | Idea | Typical behaviour here |
|--------|------|------------------------|
| **Bagging** | Average many trees on bootstrap samples → lower **variance** | RF is strong on noisy churn interactions |
| **Boosting** | Sequentially fix hard mistakes → lower **bias** | GB / XGB / LGBM often lift AUC; AdaBoost can be noisier |
| **Stacking** | Meta-learner blends base probabilities | Wins when bases make *different* errors |
| **Hybrid RF+ANN** | Soft-vote tree rules + neural surfaces | Stabilises edge cases neither model owns alone |

### Why models behave this way (Sri Lanka telecom story)
1. **Support calls + network drops** dominate — quality pain drives prepaid users away faster than age or SMS volume.
2. **Competitor offers** matter in a 4-operator market (Dialog, Mobitel, Hutch, Airtel).
3. **Family bundles, contracts, OTT add-ons** act as retention anchors (negative churn pressure).
4. **Long gaps since top-up** on prepaid lines signal silent attrition before formal churn.
5. Bagging (RF) captures these **feature interactions**; boosting focuses on **borderline** customers; stacking / hybrid voting combine complementary strengths.

### Recommendation for a retention team
Deploy the **best ROC-AUC model** from the table above for ranking who to call first, and use RF feature importances to design offers (data bonuses, network fixes, bundle discounts) rather than only discounting bills.


## 9. Persist models for the dashboard

In [ ]:
# Save best models for Streamlit (joblib / pickle-compatible)
safe_map = {
    'Bagging (Trees)': 'bagging_trees.joblib',
    'Random Forest (Bagging)': 'random_forest_bagging.joblib',
    'AdaBoost (Boosting)': 'adaboost_boosting.joblib',
    'Gradient Boosting': 'gradient_boosting.joblib',
    'XGBoost (Boosting)': 'xgboost_boosting.joblib',
    'LightGBM (Boosting)': 'lightgbm_boosting.joblib',
    'Stacking (RF+GB+Ada->LR)': 'stacking_rf_gb_ada_to_lr.joblib',
    'Hybrid RF+ANN Voting': 'hybrid_rf_ann_voting.joblib',
}

export_metrics = {}
for name, res in results.items():
    export_metrics[name] = {m: float(res[m]) for m in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']}
    fname = safe_map.get(name)
    if fname:
        joblib.dump(res['model'], MODEL_DIR / fname)

(MODEL_DIR / 'metrics.json').write_text(json.dumps(export_metrics, indent=2), encoding='utf-8')
print('Saved to', MODEL_DIR)
metrics_df


## How to open the dashboard

**Live hosted app:** https://sl-mobile-retention-desk.streamlit.app/

**Local / VS Code:**
`ash
streamlit run dashboard/app.py
`

**GitHub:** https://github.com/DulshanRavidu/sl-mobile-retention-desk

### Video reminder (≤ 3 min)
1. Hook — churn in Sri Lankan telecom  
2. Show local features  
3. Show bagging vs boosting vs stacking vs hybrid comparison  
4. Explain *why* AdaBoost leads ROC-AUC and Stacking leads recall/F1  
5. Live prediction demo  
6. Close with AI disclosure + GitHub / Streamlit links
